# 94 — Site Geospatial Fusion (merge-safe replacement)

This replacement fixes the duplicate-column merge failure by prefixing overlapping
context columns from news, population, and health layers before merging.

In [ ]:
import os, re, json, hashlib, platform, sys, math
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {"project": "AirQuality26", "step": step_name, "created_at_utc": utcnow(), "platform": {"python": sys.version, "platform": platform.platform()}, "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()}, "artifacts": [], "notes": []}

def add_artifact(manifest, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

In [ ]:
step = "94_site_geospatial_fusion"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

sites, _ = safe_read_csv(OUTPUTS / "88_site_coordinate_registry" / "site_coordinate_registry.csv")
ground, _ = safe_read_csv(OUTPUTS / "89_ground_monitor_radius_linkage" / "ground_monitor_radius_linkage.csv")
met, _ = safe_read_csv(OUTPUTS / "90_meteorology_radius_linkage" / "meteorology_radius_linkage.csv")
news, _ = safe_read_csv(OUTPUTS / "91_news_radius_context" / "news_radius_context.csv")
pop, _ = safe_read_csv(OUTPUTS / "92_population_settlement_linkage" / "population_settlement_linkage.csv")
health, _ = safe_read_csv(OUTPUTS / "93_health_context_layer" / "health_context_layer.csv")

if sites.empty:
    raise FileNotFoundError("Need outputs/88_site_coordinate_registry/site_coordinate_registry.csv")

base = sites.copy()

if not ground.empty:
    g = ground.groupby("site_key", as_index=False).agg(
        ground_monitor_count=("monitor_name","size"),
        nearest_ground_monitor_distance_miles=("distance_miles","min")
    )
    base = base.merge(g, on="site_key", how="left")

if not met.empty:
    m = met.groupby("site_key", as_index=False).agg(
        weather_station_count=("weather_station_name","size"),
        nearest_weather_station_distance_miles=("distance_miles","min")
    )
    base = base.merge(m, on="site_key", how="left")

def merge_context(base_df, ctx_df, cols, prefix):
    if ctx_df.empty:
        return base_df
    keep = [c for c in cols if c in ctx_df.columns]
    if "site_key" not in keep:
        keep = ["site_key"] + keep
    ctx = ctx_df[keep].drop_duplicates().copy()
    rename = {}
    for c in ctx.columns:
        if c != "site_key" and c in base_df.columns:
            rename[c] = f"{prefix}_{c}"
    ctx = ctx.rename(columns=rename)
    return base_df.merge(ctx, on="site_key", how="left")

base = merge_context(base, news, ["nearest_town","nearest_city","nearby_places","suggested_news_query_terms"], "news")
base = merge_context(base, pop, ["nearest_town","nearest_city","settlement_count_listed","settlement_list","population_context_flag"], "pop")
base = merge_context(base, health, ["nearest_town","nearest_city","health_context_status","health_context_note","suggested_public_health_indicators"], "health")

for c in ["ground_monitor_count","nearest_ground_monitor_distance_miles","weather_station_count","nearest_weather_station_distance_miles"]:
    if c not in base.columns:
        base[c] = 0 if c.endswith("count") else np.nan

nearest_city_series = base["nearest_city"] if "nearest_city" in base.columns else pd.Series("", index=base.index)

base["geospatial_confidence_score"] = (
    np.where(base["lat"].notna() & base["lon"].notna(), 1.0, 0.0)
    + np.where(pd.to_numeric(base["ground_monitor_count"], errors="coerce").fillna(0) > 0, 1.0, 0.0)
    + np.where(pd.to_numeric(base["weather_station_count"], errors="coerce").fillna(0) > 0, 1.0, 0.0)
    + np.where(nearest_city_series.astype(str).str.len() > 0, 0.5, 0.0)
).round(2)

base.to_csv(out / "site_geospatial_fusion.csv", index=False)
write_json(out / "site_geospatial_fusion_summary.json", {
    "rows": int(len(base)),
    "sites_with_coordinates": int(base["lat"].notna().sum()),
    "sites_with_ground_candidates": int((pd.to_numeric(base["ground_monitor_count"], errors="coerce").fillna(0) > 0).sum()),
    "sites_with_weather_candidates": int((pd.to_numeric(base["weather_station_count"], errors="coerce").fillna(0) > 0).sum()),
})
manifest = manifest_base(step, [CONFIGS / "phase88_94_geospatial.yml", CONFIGS / "site_geospatial_registry.yml"])
add_artifact(manifest, out / "site_geospatial_fusion.csv")
add_artifact(manifest, out / "site_geospatial_fusion_summary.json")
write_json(out / "manifest.json", manifest)
display(base.head(20))